
# Univariate Linear Regression — California Housing

## Objective

This notebook develops a univariate linear regression model to predict median house value using median income.

**Research Question:** Can median income predict the median house value in California census block groups?

- Predictor (X): `MedInc` — Median Income
- Target (y): `MedHouseVal` — Median House Value
- Dataset: California Housing Dataset

Two modelling approaches will be implemented:

1. Linear Regression from Scratch using Gradient Descent.
2. Linear Regression using scikit-learn.

Both implementations will be compared using RMSE, MAE, R², and regression visualizations.


In [1]:

# Import libraries for data handling and preprocessing

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("All required libraries imported successfully.")


All required libraries imported successfully.



## 1. Data Loading

The California Housing Dataset was previously collected and saved as a CSV file during the data-sourcing stage.

We load the saved CSV rather than download it again, helping keep the experiment reproducible.


In [2]:

# Load the California dataset from the raw data folder

df = pd.read_csv("../data/raw/california_housing.csv")

print("Dataset shape:", df.shape)

df.head()


Dataset shape: (20640, 9)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422



## 2. Feature Selection

For univariate linear regression, only one input variable is selected.

- X = `MedInc`
- y = `MedHouseVal`

The model will learn the relationship between median income and median house value.


In [3]:

# Select the predictor and target variable

X = df[["MedInc"]]
y = df["MedHouseVal"]

print("Predictor shape:", X.shape)
print("Target shape:", y.shape)

display(X.head())
display(y.head())


Predictor shape: (20640, 1)
Target shape: (20640,)


,MedInc
0,8.3252
1,8.3014
2,7.2574
3,5.6431
4,3.8462


0    4.526
1    3.585
2    3.521
3    3.413
4    3.422
Name: MedHouseVal, dtype: float64


## 3. Data Cleaning

Before model training, we check whether the selected predictor or target contains missing values.

Observations with missing predictor or target values would be excluded from the modelling dataset. The original CSV remains unchanged.


In [4]:

# Create a modelling dataset using our selected variables

model_data = df[["MedInc", "MedHouseVal"]].copy()

# Check missing values before cleaning
print("Missing values before cleaning:")
print(model_data.isnull().sum())

# Remove missing predictor or target values, if any
model_data = model_data.dropna(
    subset=["MedInc", "MedHouseVal"]
)

# Separate X and y again after cleaning
X = model_data[["MedInc"]]
y = model_data["MedHouseVal"]

print("\nDataset shape after cleaning:", model_data.shape)

print("\nMissing values after cleaning:")
print(model_data.isnull().sum())


Missing values before cleaning:
MedInc         0
MedHouseVal    0
dtype: int64

Dataset shape after cleaning: (20640, 2)

Missing values after cleaning:
MedInc         0
MedHouseVal    0
dtype: int64



## 4. Train/Test Split

The dataset is divided into two subsets:

- Training data: 80%
- Testing data: 20%

The training subset is used to fit the regression model, while the testing subset is reserved for evaluating its performance on unseen observations.

A fixed random state of 42 ensures that the same data split can be reproduced.


In [5]:

# Split the dataset into training and testing subsets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


X_train: (16512, 1)
X_test: (4128, 1)
y_train: (16512,)
y_test: (4128,)



## 5. Feature Standardization

Standardization transforms the predictor so that it has a mean close to zero and a standard deviation close to one.

The StandardScaler is fitted only on the training data and then applied to both training and testing data to prevent data leakage.

The target variable is kept in its original units for easier interpretation of model predictions and evaluation metrics.


In [6]:

# Initialize StandardScaler
scaler = StandardScaler()

# Fit and transform the training predictor
X_train_scaled = scaler.fit_transform(X_train)

# Transform the testing predictor using the same scaler
X_test_scaled = scaler.transform(X_test)

# Verify the standardized training data
print("Training mean:", round(X_train_scaled.mean(), 4))
print("Training standard deviation:", round(X_train_scaled.std(), 4))

print("\nFirst five standardized training values:")
print(X_train_scaled[:5])


Training mean: -0.0
Training standard deviation: 1.0

First five standardized training values:
[[-0.326196  ]
 [-0.03584338]
 [ 0.14470145]
 [-1.01786438]
 [-0.17148831]]



## 6. Linear Regression From Scratch

### Hypothesis Function

Univariate linear regression predicts a target value using one input feature.

The mathematical equation is:

$$
h_\theta(x) = \theta_0 + \theta_1 x
$$

Where:

- $h_\theta(x)$ is the predicted median house value.
- $x$ is the standardized median income.
- $\theta_0$ is the intercept.
- $\theta_1$ is the slope.

Our objective is to find the optimal values of $\theta_0$ and $\theta_1$ using gradient descent.


In [7]:

# Hypothesis function for univariate linear regression

def predict_from_scratch(x, theta_0, theta_1):
    return theta_0 + theta_1 * x



### Mean Squared Error (MSE) Cost Function

Mean Squared Error measures the average squared difference between actual and predicted values.

$$
MSE = \frac{1}{m}\sum_{i=1}^{m}(y_i-\hat{y}_i)^2
$$

Where:

- $m$ is the number of training observations.
- $y_i$ is the actual house value.
- $\hat{y}_i$ is the predicted house value.

The goal of gradient descent is to minimize this cost.
